In [19]:
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import pairwise_distances, precision_score, recall_score, f1_score, accuracy_score
from joblib import Parallel, delayed
from collections import Counter

def minkowski_distance(x, y, p):
    return np.sum(np.abs(x - y) ** p) ** (1 / p)

def run_model(X_train, y_train, X_test, y_test, beta, alpha):
    n, m = X_train.shape

    # 使用向量化计算距离矩阵
    dis_arr = pairwise_distances(X_train, metric='euclidean')
    distances = dis_arr[np.triu_indices_from(dis_arr, k=1)]
    delta = np.percentile(distances, beta)

    # 初始化结果列表
    neighborhoods = []
    neighborhoods_label = []
    granules = []

    # 并行处理每个训练样本
    def process_training_sample(i):
        # 找到与样本 i 距离小于 delta 的所有样本索引
        neighbors = np.where(dis_arr[i] < delta)[0]
        while len(neighbors) > 1:
            pairwise_in_neighborhood = dis_arr[np.ix_(neighbors, neighbors)]
            max_pairwise_distance = np.max(pairwise_in_neighborhood)
            if max_pairwise_distance < delta:
                break  # Condition met
            else:
                # Reduce delta dynamically
                current_delta = np.sort(dis_arr[i, neighbors])[-2]
                neighbors = np.where(dis_arr[i] <= current_delta)[0]

        if len(neighbors) == 0:
            return None
        
        temp_label = y_train[neighbors]
        count = pd.Series(temp_label).value_counts()
        if not count[count > alpha * len(neighbors)].empty:
            label = count[count > alpha * len(neighbors)].index.tolist()[0]
            return i, neighbors, label
        else:
            return None

    # 使用多线程并行处理
    results = Parallel(n_jobs=-1)(delayed(process_training_sample)(i) for i in range(n))

    # 过滤结果
    for res in results:
        if res is not None:
            i, neighbors, label = res
            neighborhoods.append(i)
            granules.append(neighbors)
            neighborhoods_label.append(label)

    # 将 neighborhoods 转换为 NumPy 数组以便于索引
    neighborhoods = np.array(neighborhoods)

    # 初始化变量以收集预测结果和真实标签
    y_true = []
    y_pred = []

    # 并行处理测试样本
    def process_test_sample(i, t):
        # 计算测试样本与所有训练样本的距离
        dis_to_train = np.linalg.norm(X_train - t, axis=1)
        # 找到距离小于 delta 的训练样本索引
        mask = dis_to_train[neighborhoods] < delta
        if np.any(mask):
            close_indices = neighborhoods[mask]
            reduced_close = []
            for i, close_index in enumerate(close_indices):
                distances_to_granule = dis_to_train[granules[i]]
                if np.max(distances_to_granule) < delta:
                    reduced_close.append(close_index)
            mc_labels = y_train[close_indices].tolist()
            if len(mc_labels) > 0:
                # 计算标签的频率
                label_counts = Counter(mc_labels)

                # 找到频率大于 50% 的标签
                majority_label, count = max(label_counts.items(), key=lambda x: x[1])
                if count > 0.9 * len(mc_labels):
                    return y_test[i], majority_label  # 返回多数标签
                else:
                    return y_test[i], None
            else:
                return y_test[i], None  # 无法预测时，返回 None
        else:
            return y_test[i], None


    # 使用多线程并行处理测试样本
    test_results = Parallel(n_jobs=8)(delayed(process_test_sample)(i, t) for i, t in enumerate(X_test))

    # 收集预测结果和真实标签
    for true_label, pred_label in test_results:
        y_true.append(true_label)
        y_pred.append(pred_label)

    # 过滤无法预测的样本
    y_true_filtered = []
    y_pred_filtered = []
    for true_label, pred_label in zip(y_true, y_pred):
        if pred_label is not None:
            y_true_filtered.append(true_label)
            y_pred_filtered.append(pred_label)

    # 计算评价指标
    if y_pred_filtered:
        coverage = len(y_pred_filtered) / len(y_test)
        accuracy = accuracy_score(y_true_filtered, y_pred_filtered)
        precision = precision_score(y_true_filtered, y_pred_filtered, average='weighted', zero_division=0)
        recall = accuracy * coverage
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        accuracy = 0
        precision = 0
        recall = 0
        f1 = 0
        coverage = 0

    # 计算覆盖率

    return accuracy, precision, recall, f1, coverage

In [ ]:
import numpy as np
import pandas as pd
import random
import math

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

from joblib import Parallel, delayed

# from twdneighborhood import run_model
# from twdnp import run_model
# from twdnpchev import run_model

# 加载数据
data = pd.read_csv('banknote.csv')

y = data.iloc[:, -1].to_numpy()
scaler = MinMaxScaler()
X = scaler.fit_transform(data.iloc[:, :-1])

# 设置运行次数
n_runs = 1  # 或者 20，根据您的需要

# 设置 delta 和 alpha
m = X.shape[1]
# delta = 0.03 * math.sqrt(m)
beta = 3
alpha = 0.80 # 您可以根据需要调整 alpha 值

# 存储结果
accuracies = []
precisions = []
recalls = []
f1_scores = []
coverages = []

for run in range(n_runs):
    # 随机划分数据集
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random.randint(0, 42+n_runs+run))  # 设置 random_state 以确保可重复性

    # 调用模型
    accuracy, precision, recall, f1, coverage = run_model(X_train, y_train, X_test, y_test, beta, alpha)

    # 存储结果
    accuracies.append(accuracy)
    precisions.append(precision)
    recalls.append(recall)
    f1_scores.append(f1)
    coverages.append(coverage)

# 计算平均值和标准差
mean_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

mean_precision = np.mean(precisions)
std_precision = np.std(precisions)

mean_recall = np.mean(recalls)
std_recall = np.std(recalls)

mean_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)

mean_coverage = np.mean(coverages)
std_coverage = np.std(coverages)

# 输出结果
print(f'Accuracy: {mean_accuracy:.4f}±{std_accuracy:.4f}')
print(f'Precision: {mean_precision:.4f}±{std_precision:.4f}')
print(f'Recall: {mean_recall:.4f}±{std_recall:.4f}')
print(f'F1-score: {mean_f1:.4f}±{std_f1:.4f}')
print(f'Coverage: {mean_coverage:.4f}±{std_coverage:.4f}')

In [ ]:
from itertools import product

def hyperparameter_tuning(beta_values, alpha_values, X_train, y_train, X_test, y_test):
    """
    对模型进行超参数调优，搜索最佳的 beta 和 alpha 组合。
    
    参数:
    - beta_values: beta 的候选值列表
    - alpha_values: alpha 的候选值列表
    - 其他参数: 数据集
    """
    param_combinations = list(product(beta_values, alpha_values))
    results = [run_model(X_train, y_train, X_test, y_test, beta, alpha) for beta, alpha in param_combinations]
    results_df = pd.DataFrame(results)
    
    return results_df



# 读取并预处理数据
data = pd.read_csv('sonar.csv')

y = data.iloc[:, -1].to_numpy()
scaler = MinMaxScaler()
X = scaler.fit_transform(data.iloc[:, :-1])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 定义超参数搜索范围
beta_values = np.arange(0.1, 5.1, 0.1).tolist()  # [0.1, 0.2, ..., 5.0]
alpha_values = np.arange(0.5, 1.0, 0.05).tolist()   # [0.5, 0.55, ..., 0.95]

print("开始超参数调优...")
results_df = hyperparameter_tuning(beta_values, alpha_values, X_train, y_train, X_test, y_test)

print("\n所有超参数组合的评价指标:")
print(results_df)

# 找到最佳的超参数组合，基于 F1 分数
best_result = results_df.loc[results_df['f1_score'].idxmax()]
print("\n最佳超参数组合:")
print(best_result)

# 详细展示最佳组合的指标
print(f"\n最佳组合的详细指标:\nBeta: {best_result['beta']}\nAlpha: {best_result['alpha']}\n"
      f"Accuracy: {best_result['accuracy']:.4f}\nPrecision: {best_result['precision']:.4f}\n"
      f"Recall: {best_result['recall']:.4f}\nF1 Score: {best_result['f1_score']:.4f}\n"
      f"Coverage: {best_result['coverage']:.4f}")




开始超参数调优...


In [43]:
best_result = results_df.loc[results_df[4].idxmax()]
print("\n最佳超参数组合:")
print(best_result)



最佳超参数组合:
0    0.909091
1    0.922078
2    0.476190
3    0.628041
4    0.523810
Name: 384, dtype: float64
